## Project 2: Content-Based and Collaborative Filtering

**Course:** DATA 612 – Recommender Systems<br>
**Student:** Inna Yedzinovich

### Phase 0 — Data Acquisition and Preparation

For this project, we used a MovieLens-based dataset available at:
https://grouplens.org/datasets/movielens/ml_belief_2024/
The dataset includes:

 - user_rating_history.csv: user–movie ratings
 - movies.csv: movie metadata (titles and genres)

The ratings dataset provides the core structure required for building a recommender system, in the form:
| userId | movieId | rating |


The full dataset is large, so we created a smaller subset of approximately 50 users and 50 movies.

This was done to:

 - reduce computational complexity
 - make the system easier to debug and interpret
 - maintain sufficient data density for similarity calculations

Instead of randomly sampling data, we selected:

 - users with a high number of ratings
 - movies that were rated frequently

This ensures the resulting user–item matrix is not too sparse.

Data preparation was performed using Python (pandas), with assistance from Copilot.


In [1]:
# imports
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error, mean_absolute_error



In [2]:
ratings = pd.read_csv("data/user_rating_history.csv")
movies = pd.read_csv("data/movies.csv")

# Select users with many ratings
user_counts = ratings['userId'].value_counts()
active_users = user_counts[user_counts >= 50].index[:50]

subset = ratings[ratings['userId'].isin(active_users)]

# Select frequently rated movies
movie_counts = subset['movieId'].value_counts()
popular_movies = movie_counts[movie_counts >= 50].index[:50]

subset = subset[subset['movieId'].isin(popular_movies)]

# Fix duplicate issue with pivot_table
matrix = subset.pivot_table(
    index='userId',
    columns='movieId',
    values='rating',
    aggfunc='mean'
)

print("Matrix shape:", matrix.shape)
print(matrix.head())

Matrix shape: (50, 50)
movieId  111     260     356     541       924     1196    1197    1198    \
userId                                                                      
55083       NaN     4.5    4.75     NaN  3.833333     3.5     4.0    3.25   
102175      4.5     4.0    4.50     4.5  2.000000     4.0     4.0    4.00   
114150      5.0     5.0    4.75     3.0  2.000000     5.0     5.0    5.00   
121987      2.5     3.0    2.50     3.0  3.000000     4.0     4.0    3.00   
125273      5.0     5.0    5.00     5.0  5.000000     5.0     5.0    5.00   

movieId  1200    1206    ...  168250  170705  177615  179401  182715  187593  \
userId                   ...                                                   
55083       NaN     NaN  ...     NaN    4.25     NaN     NaN     NaN     NaN   
102175      4.0     4.5  ...     NaN    4.50     3.5     2.0     NaN     NaN   
114150      5.0     4.0  ...     4.0    4.50     NaN     3.5     1.5     4.5   
121987      3.0     4.0  ...     5.0 

### Phase 1 — User–User Collaborative Filtering

In this phase, we implement User–User Collaborative Filtering. 

The idea is that users with similar rating behavior will have similar preferences, and we use this similarity to predict missing ratings. 

In recommender systems, missing ratings represent unknown information and should not be replaced with zeros, as this would introduce artificial similarity between users who have not rated the same items. Instead, we normalize the data by subtracting each user’s average rating, which removes user-specific rating bias and ensures that similarity is based on relative preferences (similar to what was done in Project 1). After normalization, missing values are filled with zeros, which now represent no deviation from the user’s average rating rather than actual ratings. Cosine similarity is then computed between these normalized user vectors. Finally, predicted ratings are calculated as a weighted combination of ratings from similar users, and the user’s average rating is added back to obtain the final predictions.

What we are doing:

- Adjust ratings by removing each user’s average
- Represent each user as a vector of ratings
- Find users with similar rating patterns
- Measure similarity using cosine similarity
- Use similar users to estimate missing ratings
- Predict ratings by combining information from similar users

In [3]:
# 1: Computer User's mean
user_means = matrix.mean(axis=1)
print(user_means.head())

# 2: Mean-centering = subtract each user’s average rating from their ratings
matrix_centered = matrix.sub(user_means, axis=0)
print("Centered matrix:")
print(matrix_centered.head())

# 3: Fill NaN AFTER centering
# (0 = no deviation from user's average)
matrix_centered_filled = matrix_centered.fillna(0)
print("Centered matrix filled:")
print(matrix_centered_filled.head())

# 4: Compute cosine similarity (how similar two users are based on their rating patterns)
user_similarity = cosine_similarity(matrix_centered_filled)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=matrix.index,
    columns=matrix.index
)

print("\nUser similarity matrix:")
print(user_similarity_df.head())

# 5: Ratings Prediction
predicted_centered = user_similarity_df.dot(matrix_centered_filled) / \
                     user_similarity_df.sum(axis=1).values.reshape(-1, 1)

# Add user mean back
predicted_user = predicted_centered.add(user_means, axis=0)

print("\nPredicted ratings:")
print(predicted_user.head())

userId
55083     3.626736
102175    3.616279
114150    3.934783
121987    3.515625
125273    3.931818
dtype: float64
Centered matrix:
movieId    111       260       356       541       924       1196      1197    \
userId                                                                          
55083         NaN  0.873264  1.123264       NaN  0.206597 -0.126736  0.373264   
102175   0.883721  0.383721  0.883721  0.883721 -1.616279  0.383721  0.383721   
114150   1.065217  1.065217  0.815217 -0.934783 -1.934783  1.065217  1.065217   
121987  -1.015625 -0.515625 -1.015625 -0.515625 -0.515625  0.484375  0.484375   
125273   1.068182  1.068182  1.068182  1.068182  1.068182  1.068182  1.068182   

movieId    1198      1200      1206    ...    168250    170705    177615  \
userId                                 ...                                 
55083   -0.376736       NaN       NaN  ...       NaN  0.623264       NaN   
102175   0.383721  0.383721  0.883721  ...       NaN  0.883721 -0.1162

In this step, we evaluate how well our model predicts ratings using a train/test split, as we did in project 1. We want to make sure the model is tested on unseen data. We calculate RMSE and MAE to measure prediction accuracy, which helps us understand how close the predicted ratings are to the actual ratings.

In [4]:
# train/test
np.random.seed(66)

train = matrix.copy()
test = matrix.copy()

# hide 20% of known ratings
mask = ~matrix.isna()
random_mask = np.random.rand(*matrix.shape) < 0.2

test[~(mask & random_mask)] = np.nan
train[mask & random_mask] = np.nan


# train model
user_means = train.mean(axis=1)
train_centered = train.sub(user_means, axis=0)
train_centered_filled = train_centered.fillna(0)

user_similarity = cosine_similarity(train_centered_filled)

predicted_centered = user_similarity.dot(train_centered_filled) / \
                     np.abs(user_similarity).sum(axis=1).reshape(-1, 1)

predicted_user = predicted_centered + user_means.values.reshape(-1, 1)

predicted_user = pd.DataFrame(
    predicted_user,
    index=train.index,
    columns=train.columns
)

# RMSE + MAE
mask = ~test.isna()

y_true = test.values[mask.values]
y_pred = predicted_user.values[mask.values]

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)

RMSE: 1.5157169665726071
MAE: 1.1157784772680188


The model achieved an RMSE of about 1.52 and an MAE of about 1.12, meaning that predicted ratings are on average about one rating point away from the true values. This level of error is reasonable for a basic collaborative filtering model, since predictions are based on averaging similar users’ ratings, which tends to smooth extreme preferences and reduces accuracy for very high or very low ratings.

### Phase 2 — Content-Based Filtering

In this phase, we implement Content-Based Filtering.

The idea is that movies with similar features (such as genres) will be similar to each other, and we use this similarity to recommend movies that are similar to those a user already likes. Instead of relying on other users, this approach focuses on item characteristics to make predictions. In short, if two movies have similar features, and a user liked one, they will like the other. 

Each movie is represented using its genre information. We first clean and transform the genre text into a usable format, and then convert it into numerical feature vectors. This allows each movie to be represented as a vector of its genre attributes. Cosine similarity is then computed between these movie vectors to measure how similar movies are based on their features.

Using this similarity, predicted ratings are calculated as a weighted combination of ratings for similar movies that a user has already rated. This is implemented using matrix multiplication, where each prediction is based on the similarity between items. Similar to Phase 1, we use a train/test split to evaluate the model on unseen data, and compute RMSE and MAE to measure prediction accuracy.

What we are doing:

- Clean genre text formatting
- Convert text into feature vectors
- Represent movies as numeric genre data
- Compute movie-to-movie similarity
- Measure similarity using cosine similarity
- Store similarities in a labeled matrix

In [5]:
# 1: Prepare movie features

# ensure datatype types match
matrix.columns = matrix.columns.astype(int)
movies['movieId'] = movies['movieId'].astype(int)

# keep only relevant movies and clean up
movies_small = movies[movies['movieId'].isin(matrix.columns)].copy()
movies_small['genres_clean'] = movies_small['genres'].str.replace('|', ' ', regex=False)
print(movies_small.head())


# 2: Convert genres to vectors (based on the article logic in http://infolab.stanford.edu/~ullman/mmds/ch9.pdf)
vectorizer = CountVectorizer()
genre_matrix = vectorizer.fit_transform(movies_small['genres_clean'])
print(genre_matrix.shape)
print(genre_matrix)

genre_df = pd.DataFrame(
    genre_matrix.toarray(),
    index=movies_small['movieId']
)
print(genre_df.head())


# 3: Compute item-item similarity (same idea as in phase 1)

movie_similarity = cosine_similarity(genre_matrix)
movie_similarity_df = pd.DataFrame(
    movie_similarity,
    index=movies_small['movieId'],
    columns=movies_small['movieId']
)
print(movie_similarity_df.head())

     movieId                                      title  \
109      111                         Taxi Driver (1976)   
257      260  Star Wars: Episode IV - A New Hope (1977)   
351      356                        Forrest Gump (1994)   
536      541                        Blade Runner (1982)   
903      924               2001: A Space Odyssey (1968)   

                       genres              genres_clean  
109      Crime|Drama|Thriller      Crime Drama Thriller  
257   Action|Adventure|Sci-Fi   Action Adventure Sci-Fi  
351  Comedy|Drama|Romance|War  Comedy Drama Romance War  
536    Action|Sci-Fi|Thriller    Action Sci-Fi Thriller  
903    Adventure|Drama|Sci-Fi    Adventure Drama Sci-Fi  
(50, 18)
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 172 stored elements and shape (50, 18)>
  Coords	Values
  (0, 5)	1
  (0, 7)	1
  (0, 15)	1
  (1, 0)	1
  (1, 1)	1
  (1, 14)	1
  (1, 9)	1
  (2, 7)	1
  (2, 4)	1
  (2, 13)	1
  (2, 16)	1
  (3, 15)	1
  (3, 0)	1
  (3, 14)	1
  (3, 9)	1
 

Similar to Phase 1, a train/test split is applied to ensure that the content-based model is evaluated on unseen data, and RMSE and MAE are computed to measure prediction accuracy.

In [6]:
# Train/Test
np.random.seed(66)

train = matrix.copy()
test = matrix.copy()

mask = ~matrix.isna()
random_mask = np.random.rand(*matrix.shape) < 0.2

test[~(mask & random_mask)] = np.nan
train[mask & random_mask] = np.nan

# Prediction: Prediction = (ratings × similarity) / sum(|similarity|)
common_movies = train.columns.intersection(movie_similarity_df.index)

train_cb = train[common_movies]
movie_sim_cb = movie_similarity_df.loc[common_movies, common_movies]

train_cb_filled = train_cb.fillna(0)

predicted_cb = train_cb_filled.dot(movie_sim_cb) / \
               np.abs(movie_sim_cb).sum(axis=1).values.reshape(1, -1)             
print(predicted_cb.head())

# RMSE/MAE 
mask = ~test[common_movies].isna()

y_true = test[common_movies].values[mask.values]
y_pred = predicted_cb.values[mask.values]

rmse_cb = np.sqrt(mean_squared_error(y_true, y_pred))
mae_cb = mean_absolute_error(y_true, y_pred)

print("Content RMSE:", rmse_cb)
print("Content MAE:", mae_cb)

movieId    111       260       356       541       924       1196      1197    \
userId                                                                          
55083    1.073437  0.732534  2.039393  0.676344  0.755609  0.732534  1.370596   
102175   2.929540  3.039298  2.608942  3.131216  2.966090  3.039298  2.842516   
114150   2.485435  3.416432  3.191150  3.250036  3.143240  3.416432  3.692810   
121987   2.164468  2.809269  2.414931  2.745945  2.657858  2.809269  2.774768   
125273   3.588076  3.136377  3.224480  3.127269  3.217775  3.136377  3.237018   

movieId    1198      1200      1206    ...    168250    170705    177615  \
userId                                 ...                                 
55083    0.928740  0.733006  0.758836  ...  0.737800  1.288674  1.970057   
102175   2.919716  2.961551  3.063457  ...  2.172461  2.821819  2.620420   
114150   3.686196  3.370185  2.878657  ...  2.900796  3.390901  3.329414   
121987   2.748355  2.810460  2.580106  ...  2.822548

The model achieved an RMSE of about 1.89 and an MAE of about 1.62, meaning that predicted ratings are on average more than one rating point away from the true values. This higher level of error is expected for a basic content-based model, since it relies only on simple genre features, which do not fully capture user preferences and often lead to less accurate predictions compared to collaborative filtering.

### Model Comparison
In this phase, we compare the performance of User–User Collaborative Filtering and Content-Based Filtering using the same evaluation approach (train/test split with RMSE and MAE).

The User–User model achieved an RMSE of about 1.52 and an MAE of about 1.12, while the Content-Based model had a higher RMSE of about 1.89 and an MAE of about 1.62. This shows that the collaborative filtering approach performed better in predicting user ratings.

This difference is expected because collaborative filtering uses actual user behavior and captures patterns in how users rate movies, while the content-based model relies only on genre information, which is a much simpler representation of movies. Since genres cannot fully describe user preferences (such as actors, storyline, or personal taste), the content-based model produces less accurate predictions. This suggests that user behavior provides a stronger signal than basic item features, especially when those features are limited.

Overall, both models capture general preferences, but User–User Collaborative Filtering provides more accurate results, while Content-Based Filtering serves as a simpler baseline model. This project demonstrates how different recommendation strategies impact prediction accuracy and highlights the importance of data representation and similarity computation in recommender systems.